# MetaCal Benchmark — T-04

Isolated task notebook.

In [3]:
import re
import kaggle_benchmarks as kbench

def extract_confidence(text: str) -> int | None:
    """Pull the first integer 0-100 that follows confidence keywords."""
    # strip thinking blocks (DeepSeek-R1, Qwen thinking)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    pattern = r"(?:confidence|certain|sure)[^\d]{0,30}(\d{1,3})"
    match = re.search(pattern, text, re.IGNORECASE)
    if not match:
        nums = re.findall(r"\b(\d{1,3})\b", text)
        nums = [n for n in nums if 0 <= int(n) <= 100]
        return int(nums[-1]) if nums else None
    return int(match.group(1))


def compute_ece(confidences, correctness, n_bins=10):
    """Expected Calibration Error — lower is better."""
    bins = [[] for _ in range(n_bins)]
    for conf, correct in zip(confidences, correctness):
        idx = min(int(conf / 100 * n_bins), n_bins - 1)
        bins[idx].append((conf / 100, correct))
    ece = 0
    for b in bins:
        if b:
            avg_conf = sum(c for c, _ in b) / len(b)
            avg_acc = sum(r for _, r in b) / len(b)
            ece += abs(avg_conf - avg_acc) * len(b) / len(confidences)
    return round(ece, 4)


def compute_auroc(confidences, correctness):
    """AUROC — how well confidence predicts correctness."""
    pairs = sorted(zip(confidences, correctness), reverse=True)
    n_pos = sum(correctness)
    n_neg = len(correctness) - n_pos
    if n_pos == 0 or n_neg == 0:
        return None
    tp, fp, auc = 0, 0, 0
    prev_fp = 0
    for conf, correct in pairs:
        if correct:
            tp += 1
        else:
            fp += 1
            auc += tp * (fp - prev_fp)
            prev_fp = fp
    return round(auc / (n_pos * n_neg), 4)


def compute_meta_d_proxy(correct_confs, incorrect_confs):
    """Discrimination between correct and incorrect confidence."""
    if not correct_confs or not incorrect_confs:
        return None
    return round(
        sum(correct_confs) / len(correct_confs) -
        sum(incorrect_confs) / len(incorrect_confs), 2
    )


In [ ]:
@kbench.task(
    name="T-04: Post-Answer Error Flag",
    description=(
        "Two-turn task: model answers, then rates probability its answer is wrong (0-100). "
        "Error-flag scores should be high on incorrect answers. Computes real AUROC."
    )
)
def t04_post_answer_error_flag(llm) -> None:
    ITEMS = [
        # Questions most models answer correctly — error flag should be LOW
        ("What is the capital of Australia?",              "canberra"),
        ("What is 17 multiplied by 23?",                   "391"),
        ("Who invented the telephone?",                    "bell"),
        ("What country has the longest coastline?",        "canada"),
        ("What is the chemical formula for water?",        "h2o"),
        ("What is the square root of 169?",                "13"),
        ("How many days are in a leap year?",              "366"),
        ("What is the capital of Japan?",                  "tokyo"),
        # Harder questions — model may err; error flag should be HIGHER if wrong
        ("What is the atomic number of Osmium?",           "76"),
        ("In what year was the Treaty of Westphalia signed?", "1648"),
        ("What is the capital of Burkina Faso?",           "ouagadougou"),
        ("Who composed the opera The Rake's Progress?",    "stravinsky"),
        ("What is the cube root of 729?",                  "9"),
        ("What is the chemical formula for laughing gas?", "n2o"),
        ("In what year did the French Revolution begin?",  "1789"),
        # Items frontier models frequently get wrong — ensures AUROC has signal
        ("How many times does 24 go into 1440?",           "60"),
        ("What is the capital of Myanmar?",                "naypyidaw"),
        ("Who wrote Finnegans Wake?",                      "joyce"),
    ]
    STEP2 = (
        "On a scale of 0-100, how likely is it that your answer above is WRONG? "
        "(0 = definitely correct, 100 = definitely wrong). Give only the number."
    )
    error_flags = []
    correctness_flags = []

    for question, expected in ITEMS:
        step1 = llm.prompt(question)
        step2 = llm.prompt(STEP2)
        is_correct = expected.lower() in step1.lower()
        flag = extract_confidence(step2)
        kbench.assertions.assert_true(
            flag is not None,
            expectation=f"Model must output a numeric error-probability 0-100 for: '{question}'"
        )
        if flag is not None:
            error_flags.append(flag)
            correctness_flags.append(is_correct)

    correct_flags   = [f for f, c in zip(error_flags, correctness_flags) if c]
    incorrect_flags = [f for f, c in zip(error_flags, correctness_flags) if not c]
    if correct_flags and incorrect_flags:
        avg_correct   = sum(correct_flags) / len(correct_flags)
        avg_incorrect = sum(incorrect_flags) / len(incorrect_flags)
        kbench.assertions.assert_true(
            avg_correct < avg_incorrect,
            expectation=(
                f"AUROC-proxy: error flags should be higher on wrong answers. "
                f"Avg flag on correct={avg_correct:.1f}, incorrect={avg_incorrect:.1f}."
            )
        )

    # Compute real AUROC: does a high error flag predict an incorrect answer?
    if error_flags and len(set(correctness_flags)) == 2:
        auroc = compute_auroc(error_flags, [not c for c in correctness_flags])
        kbench.assertions.assert_true(
            auroc is not None and auroc > 0.6,
            expectation=(
                f"AUROC = {auroc}. Error flags should discriminate correct from incorrect answers "
                "(AUROC > 0.6). Value of None means all answers were correct or all were wrong."
            )
        )

    assessment = kbench.assertions.assess_response_with_judge(
        response_text=f"Error flag scores: {list(zip([q for q, _ in ITEMS], error_flags))}",
        judge_llm=kbench.judge_llm,
        criteria=[
            "Error flags on easy factual questions (capital of Australia, formula for water, etc.) should generally be low.",
            "Error flags on harder questions (Burkina Faso capital, Treaty of Westphalia, laughing gas formula) should be higher on average.",
            "The scores should show variation across items — a model giving 50 for every question is not self-monitoring.",
            "No single score should be exactly 0 or exactly 100 for all items — that indicates a degenerate strategy.",
        ]
    )
    for result in assessment.results:
        kbench.assertions.assert_true(
            result.passed,
            expectation=f"Error flag quality: {result.criterion} — {result.reason}"
        )

In [ ]:
%choose t04_post_answer_error_flag